<a href="https://colab.research.google.com/github/Jopat2409/com3610_notebooks/blob/main/huggingface_conll_2003.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Reproducing experimental results of LUKE on CoNLL-2003 Using Hugging Face Transformers

This notebook shows how to reproduce the state-of-the-art results on the [CoNLL-2003 named entity recognition dataset](https://www.clips.uantwerpen.be/conll2003/ner/) reported in [this paper](https://arxiv.org/abs/2010.01057) using the Trasnsformers library and the [fine-tuned model checkpoint](https://huggingface.co/studio-ousia/luke-large-finetuned-conll-2003) available on the Model Hub.
The source code used in the experiments is also available [here](https://github.com/studio-ousia/luke/tree/master/examples/ner).

*Currently, due to the slight difference in preprocessing, the score reproduced in this notebook is slightly lower than the score reported in the original paper (approximately 0.1 F1).*

There are two other related notebooks:

* [Reproducing experimental results of LUKE on Open Entity Using Hugging Face Transformers](https://github.com/studio-ousia/luke/blob/master/notebooks/huggingface_open_entity.ipynb)
* [Reproducing experimental results of LUKE on TACRED Using Hugging Face Transformers](https://github.com/studio-ousia/luke/blob/master/notebooks/huggingface_tacred.ipynb)

In [1]:
# Currently, LUKE is only available on the master branch
!pip install seqeval git+https://github.com/huggingface/transformers.git

  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-wny_26_w
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-wny_26_w
  Resolved https://github.com/huggingface/transformers.git to commit c0f8d055ce7a218e041e20a06946bf0baa8a7d6a
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16161 sha256=d1857eda112d7d48c896e184453f993afe50436d8c3a2ad937d5cf740df8a1e9
  Stored in directory: /root/.cache/pip/wheels/bc/92/f0/243288f899c2eacdfa8c5f9aede4c71a9bad0ee26a01dc5ead
  Created wheel for transformers: filename=transformers-4.50.0.dev0-py3-none-any.whl size=10862436 sha256=7a5256fff2a5e26b3cc7e73de4eb39f809d4e2ba24c393255

In [2]:
import unicodedata

import numpy as np
import seqeval.metrics
import spacy
import torch
from tqdm import tqdm, trange
from transformers import LukeTokenizer, LukeForEntitySpanClassification

## Loading the dataset

The test set of the CoNLL-2003 dataset (eng.testb) is placed in the current directory and loaded using `load_examples` function.

In [3]:
# Download the testb set of the CoNLL-2003 dataset
!wget https://raw.githubusercontent.com/synalp/NER/master/corpus/CoNLL-2003/eng.testb

--2025-03-04 03:05:26--  https://raw.githubusercontent.com/synalp/NER/master/corpus/CoNLL-2003/eng.testb
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 748096 (731K) [text/plain]
Saving to: ‘eng.testb’

eng.testb           100%[===================>] 730.56K  3.44MB/s    in 0.2s    

2025-03-04 03:05:27 (3.44 MB/s) - ‘eng.testb’ saved [748096/748096]



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Loading the fine-tuned model and tokenizer

We construct the model and tokenizer using the [fine-tuned model checkpoint](https://huggingface.co/studio-ousia/luke-large-finetuned-conll-2003).

In [4]:
# Load the model checkpoint
model = LukeForEntitySpanClassification.from_pretrained("studio-ousia/luke-large-finetuned-conll-2003")
model.eval()
model.to("cuda")

# Load the tokenizer
tokenizer = LukeTokenizer.from_pretrained("studio-ousia/luke-large-finetuned-conll-2003")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Some weights of the model checkpoint at studio-ousia/luke-large-finetuned-conll-2003 were not used when initializing LukeForEntitySpanClassification: ['luke.embeddings.position_ids']
- This IS expected if you are initializing LukeForEntitySpanClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing LukeForEntitySpanClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

entity_vocab.json:   0%|          | 0.00/15.3M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/33.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/1.00k [00:00<?, ?B/s]

In [2]:
def load_documents(dataset_file):
    """
    Load documents. A document
    """
    documents, words, labels, sentence_boundaries = [], [], [], []
    with open(dataset_file) as f:
        for line in f:
            line = line.rstrip()
            if line.startswith("-DOCSTART"):
                if words:
                    documents.append({"words": words, "labels": labels, "sentence_boundaries": sentence_boundaries})
                    words, labels, sentence_boundaries = [], [], []
                continue

            if not line:
                if not sentence_boundaries or len(words) != sentence_boundaries[-1]:
                    sentence_boundaries.append(len(words))
            else:
                items = line.split("\t")
                words.append(items[0])
                labels.append(items[-1])
    if words:
        documents.append({"words": words, "labels": labels, "sentence_boundaries": sentence_boundaries})

    return documents

def stretch_context_bounds(sentence_start, sentence_end, subword_lengths, max_token_length, total_words):
    context_start, context_end = sentence_start, sentence_end
    cur_length = sum(subword_lengths[context_start:context_end])
    while True:
        if context_start > 0:
            if cur_length + subword_lengths[context_start - 1] <= max_token_length:
                cur_length += subword_lengths[context_start - 1]
                context_start -= 1
            else:
                break
        if context_end < len(total_words):
            if cur_length + subword_lengths[context_end] <= max_token_length:
                cur_length += subword_lengths[context_end]
                context_end += 1
            else:
                break
    return context_start, context_end

def load_example(document, max_token_length = 510, max_mention_length = 30):
    words = document["words"]

    subword_lengths = [len(tokenizer.tokenize(w)) for w in words]
    total_subword_length = sum(subword_lengths)

    sentence_boundaries = document["sentence_boundaries"]

    # Loop through all sentences
    for i in range(len(sentence_boundaries) - 1):
        sentence_start, sentence_end = sentence_boundaries[i:i+2]
        context_start, context_end = ((0, len(words)) if total_subword_length <= max_token_length else stretch_context_bounds(sentence_start, sentence_end, subword_lengths, max_token_length))

        sentence_words = words[sentence_start:sentence_end]
        sentence_subword_lengths = subword_lengths[sentence_start:sentence_end]

        text = ""

        # Add words before sentence (context)
        for word in words[context_start:sentence_start]:
            if word[0] == "'" or (len(word) == 1 and is_punctuation(word)):
                text = text.rstrip()
            text += f"{word} "

        # Add words inside sentence including the start and end char positions
        word_start_char_positions, word_end_char_positions = [], []
        for word in sentence_words:
            if word[0] == "'" or (len(word) == 1 and is_punctuation(word)):
                text = text.rstrip()
            word_start_char_positions.append(len(text))
            text += word
            word_end_char_positions.append(len(text))
            text += " "

        # Add words after sentence (context)
        for word in words[sentence_end:context_end]:
            if word[0] == "'" or (len(word) == 1 and is_punctuation(word)):
                text = text.rstrip()
            text += f"{word} "
        text = text.rstrip()

        entity_spans, original_word_spans = [], []

        for word_start in range(len(sentence_words)):
            for word_end in range(word_start, len(sentence_words)):
                if sum(sentence_subword_lengths[word_start:word_end + 1]) <= max_mention_length:
                    entity_spans.append(
                        (word_start_char_positions[word_start], word_end_char_positions[word_end])
                    )
                    original_word_spans.append(
                        (word_start, word_end + 1)
                    )
        return dict(
                text=text,
                words=sentence_words,
                entity_spans=entity_spans,
                original_word_spans=original_word_spans,
            )


def load_examples(documents):
    examples = []

    for document in tqdm(documents):
      examples.append(load_example(document))

    return examples


def is_punctuation(char):
    cp = ord(char)
    if (cp >= 33 and cp <= 47) or (cp >= 58 and cp <= 64) or (cp >= 91 and cp <= 96) or (cp >= 123 and cp <= 126):
        return True
    cat = unicodedata.category(char)
    if cat.startswith("P"):
        return True
    return False

In [52]:
test_documents = load_documents("test.txt")
test_examples = load_examples(test_documents)

print("\n")
print(test_documents[0]["sentence_boundaries"])
print(test_documents[0]["words"])

print( sum([len(example["words"]) for example in test_examples]),  sum([len(document["words"]) for document in test_documents]))

print(sum(len(doc["labels"]) for doc in test_documents))

100%|██████████| 231/231 [00:02<00:00, 88.88it/s]



[0, 12, 14, 20, 45, 70, 112, 135, 152, 170, 198, 236, 248, 279, 295, 316, 325, 342, 362, 363, 389, 406, 419]
['SOCCER', '-', 'JAPAN', 'GET', 'LUCKY', 'WIN', ',', 'CHINA', 'IN', 'SURPRISE', 'DEFEAT', '.', 'Nadim', 'Ladki', 'AL-AIN', ',', 'United', 'Arab', 'Emirates', '1996-12-06', 'Japan', 'began', 'the', 'defence', 'of', 'their', 'Asian', 'Cup', 'title', 'with', 'a', 'lucky', '2-1', 'win', 'against', 'Syria', 'in', 'a', 'Group', 'C', 'championship', 'match', 'on', 'Friday', '.', 'But', 'China', 'saw', 'their', 'luck', 'desert', 'them', 'in', 'the', 'second', 'match', 'of', 'the', 'group', ',', 'crashing', 'to', 'a', 'surprise', '2-0', 'defeat', 'to', 'newcomers', 'Uzbekistan', '.', 'China', 'controlled', 'most', 'of', 'the', 'match', 'and', 'saw', 'several', 'chances', 'missed', 'until', 'the', '78th', 'minute', 'when', 'Uzbek', 'striker', 'Igor', 'Shkvyrin', 'took', 'advantage', 'of', 'a', 'misdirected', 'defensive', 'header', 'to', 'lob', 'the', 'ball', 'over', 'the', 'advancing', 

## Measuring performance

We classify all possible entity spans in the test set, exclude all spans classified into the `NIL` type, and greedily select a span from the remaining spans based on the logit of its predicted entity type in descending order.
Due to  minor differences in processing, the reproduced performance is slightly lower than the performance reported in the [original paper](https://arxiv.org/abs/2010.01057) (approximately 0.1 F1).

In [25]:
batch_size = 2
all_logits = []

for batch_start_idx in trange(0, len(test_examples), batch_size):
    batch_examples = test_examples[batch_start_idx:batch_start_idx + batch_size]
    texts = [example["text"] for example in batch_examples]
    entity_spans = [example["entity_spans"] for example in batch_examples]

    inputs = tokenizer(texts, entity_spans=entity_spans, return_tensors="pt", padding=True)
    inputs = inputs.to("cuda")
    with torch.no_grad():
        outputs = model(**inputs)
    all_logits.extend(outputs.logits.tolist())

100%|██████████| 1713/1713 [10:01<00:00,  2.85it/s]


In [26]:
class BERTError:

    def __init__(self, sentence, expected, predicted):
        self.sentence = sentence
        self.expected = expected
        self.predicted = predicted

        self.incorrect = [i for i, tag in enumerate(self.expected) if not self.token_matches(tag, self.predicted[i])]
        self.incorrect_mappings = [(self.expected[i], self.predicted[i]) for i in self.incorrect]

    def token_matches(self, t1, t2):
        return (t1 == t2 == "O") or (t1[2:] == t2[2:])

    def __repr__(self) -> str:
        words = self.sentence.split(' ')
        for index in self.incorrect:
            words[index] = f"[E:{self.expected[index]},P:{self.predicted[index]}]({words[index]})"
        return ' '.join(words)

    def to_error_dict(self):
      return {
          "sentence": self.sentence,
          "expected": self.expected,
          "tokens": self.sentence.split(' '),
          "predicted": self.predicted,
          "incorrect": self.incorrect
      }

In [35]:
final_labels = [label for document in test_documents for label in document["labels"]]

final_predictions = []
for example_index, example in enumerate(test_examples):
    logits = all_logits[example_index]
    max_logits = np.max(logits, axis=1)
    max_indices = np.argmax(logits, axis=1)
    original_spans = example["original_word_spans"]
    predictions = []
    for logit, index, span in zip(max_logits, max_indices, original_spans):
        if index != 0:  # the span is not NIL
            predictions.append((logit, span, model.config.id2label[index]))

    # construct an IOB2 label sequence
    predicted_sequence = ["O"] * len(example["words"])
    for _, span, label in sorted(predictions, key=lambda o: o[0], reverse=True):
        if all([o == "O" for o in predicted_sequence[span[0] : span[1]]]):
            predicted_sequence[span[0]] = "B-" + label
            if span[1] - span[0] > 1:
                predicted_sequence[span[0] + 1 : span[1]] = ["I-" + label] * (span[1] - span[0] - 1)

    final_predictions += predicted_sequence

In [28]:
print(seqeval.metrics.classification_report([final_labels], [final_predictions], digits=4))

ValueError: Found input variables with inconsistent numbers of samples:
[46504]
[46463]

## Recognizing named entities in a text

Finally, we extract named entities from a text using the [fine-tuned model](https://huggingface.co/studio-ousia/luke-large-finetuned-conll-2003). The input text is tokenized using [SpaCy](https://spacy.io/).

In [ ]:
text = "Star Wars is a film written and directed by George Lucas"
nlp = spacy.load("en_core_web_sm")
doc = nlp(text)

entity_spans = []
original_word_spans = []
for token_start in doc:
    for token_end in doc[token_start.i:]:
        entity_spans.append((token_start.idx, token_end.idx + len(token_end)))
        original_word_spans.append((token_start.i, token_end.i + 1))

inputs = tokenizer(text, entity_spans=entity_spans, return_tensors="pt", padding=True)
inputs = inputs.to("cuda")
with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits
max_logits, max_indices = logits[0].max(dim=1)

predictions = []
for logit, index, span in zip(max_logits, max_indices, original_word_spans):
    if index != 0:  # the span is not NIL
        predictions.append((logit, span, model.config.id2label[int(index)]))

# construct an IOB2 label sequence
predicted_sequence = ["O"] * len(doc)
for _, span, label in sorted(predictions, key=lambda o: o[0], reverse=True):
    if all([o == "O" for o in predicted_sequence[span[0] : span[1]]]):
        predicted_sequence[span[0]] = "B-" + label
        if span[1] - span[0] > 1:
            predicted_sequence[span[0] + 1 : span[1]] = ["I-" + label] * (span[1] - span[0] - 1)

for token, label in zip(doc, predicted_sequence):
    print(token, label)

Star B-MISC
Wars I-MISC
is O
a O
film O
written O
and O
directed O
by O
George B-PER
Lucas I-PER
